<h1>Step 1: Set up SQL in Jupyter</h1>

In [2]:
import pandas as pd
import sqlite3

# load cleaned file
df = pd.read_csv("../02_Cleaned/orders_cleaned.csv")

# create a small database file and put the data in a table called "orders"
con = sqlite3.connect("../02_Cleaned/maple.db")
df.to_sql("orders", con, index=False, if_exists="replace")

# small helper so you can run SQL easily
def sql(query):
    return pd.read_sql(query, con)

sql("SELECT * FROM orders LIMIT 5")

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,sales_channel,region,payment_method,...,shipping_cost,customer_rating,product_name,category,unit_cost,revenue,total_cost,profit,month,month_name
0,52895,2025-11-10,C2365,P107,1,65.0,10,Etsy,West,PayPal,...,8.99,4.0,Waffle Throw Blanket,Bedding,32.59,58.5,32.59,25.91,11,Nov
1,53245,2025-12-01,C2223,P117,2,58.0,0,Website,West,PayPal,...,8.99,4.0,Turkish Towel Set,Bath,29.74,116.0,59.48,56.52,12,Dec
2,50498,2025-03-13,C1645,P111,1,55.0,0,Instagram,Midwest,Credit Card,...,0.00,3.0,Cast Iron Skillet,Kitchen,25.13,55.0,25.13,29.87,3,Mar
3,51374,2025-06-17,C1210,P114,1,42.0,0,Amazon,Midwest,Gift Card,...,8.99,5.0,Ceramic Vase,Decor,20.14,42.0,20.14,21.86,6,Jun
4,51346,2025-06-13,C2142,P104,1,45.0,10,Etsy,Midwest,Credit Card,...,12.50,5.0,Candle Gift Set,Candles,14.40,40.5,14.40,26.10,6,Jun


<h1>Step 2: Answer the questions</h1>

<h1>Q1: Total revenue, profit and main KPIs</h1>

In [3]:
sql("""
SELECT
    ROUND(SUM(revenue), 2)                    AS total_revenue,
    ROUND(SUM(profit), 2)                     AS total_profit,
    ROUND(SUM(profit) * 100.0 / SUM(revenue), 1) AS profit_margin_pct,
    COUNT(*)                                  AS total_orders,
    ROUND(SUM(revenue) / COUNT(*), 2)         AS avg_order_value
FROM orders
WHERE order_status = 'Delivered'
""")

,total_revenue,total_profit,profit_margin_pct,total_orders,avg_order_value
0,218550.25,110606.64,50.6,3459,63.18


<h1>Q2: Monthly trend</h1>

sql("""
SELECT month, month_name,
       ROUND(SUM(revenue), 2) AS revenue,
       COUNT(*)               AS orders
FROM orders
WHERE order_status = 'Delivered'
GROUP BY month, month_name
ORDER BY month
""")

<h1>Q3a: Profit by category </h1>

In [6]:
sql("""
SELECT category,
       ROUND(SUM(revenue), 2)                       AS revenue,
       ROUND(SUM(profit), 2)                        AS profit,
       ROUND(SUM(profit) * 100.0 / SUM(revenue), 1) AS margin_pct
FROM orders
WHERE order_status = 'Delivered'
GROUP BY category
ORDER BY profit DESC
""")

,category,revenue,profit,margin_pct
0,Bedding,64766.70,28591.64,44.1
1,Kitchen,47481.15,24290.10,51.2
2,Decor,43973.95,23183.86,52.7
3,Candles,32410.90,20976.76,64.7
4,Bath,29917.55,13564.28,45.3


<h1>3b: Top 10 products by profit</h1>

In [7]:
sql("""
SELECT product_name, category,
       ROUND(SUM(profit), 2) AS profit
FROM orders
WHERE order_status = 'Delivered'
GROUP BY product_name, category
ORDER BY profit DESC
LIMIT 10
""")

,product_name,category,profit
0,Waffle Throw Blanket,Bedding,9253.65
1,Cotton Sheet Set,Bedding,8366.38
2,Linen Duvet Cover,Bedding,8271.63
3,Turkish Towel Set,Bath,8236.26
4,Ceramic Mug Set,Kitchen,7547.30
5,Rattan Mirror,Decor,7152.79
6,Candle Gift Set,Candles,6696.00
7,Woven Wall Hanging,Decor,6410.79
8,Glass Storage Jars,Kitchen,6360.62
9,Cast Iron Skillet,Kitchen,6268.70


<h1>Q4: Best sales channel</h1>

In [8]:
sql("""
SELECT sales_channel,
       COUNT(*)                          AS orders,
       ROUND(SUM(revenue), 2)            AS revenue,
       ROUND(SUM(profit), 2)             AS profit,
       ROUND(SUM(revenue) / COUNT(*), 2) AS avg_order_value
FROM orders
WHERE order_status = 'Delivered'
GROUP BY sales_channel
ORDER BY revenue DESC
""")

,sales_channel,orders,revenue,profit,avg_order_value
0,Website,1592,98803.75,50064.65,62.06
1,Etsy,694,44872.10,22550.47,64.66
2,Instagram,658,42657.40,21653.30,64.83
3,Amazon,515,32217.00,16338.22,62.56


<h1>Q5: Regions</h1>

In [9]:
sql("""
SELECT region,
       COUNT(*)               AS orders,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(SUM(profit), 2)  AS profit
FROM orders
WHERE order_status = 'Delivered'
GROUP BY region
ORDER BY revenue DESC
""")

,region,orders,revenue,profit
0,Northeast,928,57109.95,28881.95
1,West,863,54628.05,27572.18
2,Southeast,674,43126.05,21767.90
3,Midwest,584,37360.45,18936.04
4,Southwest,342,21811.80,11101.93
5,Unknown,68,4513.95,2346.64


<h1>Q6: Return rate by category</h1>

In [10]:
sql("""
SELECT category,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN order_status = 'Returned' THEN 1 ELSE 0 END) AS returned_orders,
       ROUND(SUM(CASE WHEN order_status = 'Returned' THEN 1 ELSE 0 END) * 100.0
             / COUNT(*), 1) AS return_rate_pct
FROM orders
WHERE order_status <> 'Cancelled'
GROUP BY category
ORDER BY return_rate_pct DESC
""")

,category,total_orders,returned_orders,return_rate_pct
0,Bedding,703,94,13.4
1,Kitchen,877,48,5.5
2,Candles,1036,57,5.5
3,Bath,547,28,5.1
4,Decor,549,26,4.7


<h1>Q7: Do discounts help?</h1>

In [11]:
sql("""
SELECT discount_pct,
       COUNT(*)                                     AS orders,
       ROUND(AVG(revenue), 2)                       AS avg_order_value,
       ROUND(SUM(profit) * 100.0 / SUM(revenue), 1) AS margin_pct
FROM orders
WHERE order_status = 'Delivered'
GROUP BY discount_pct
ORDER BY discount_pct
""")

,discount_pct,orders,avg_order_value,margin_pct
0,0,1849,66.97,53.0
1,5,497,63.95,50.8
2,10,515,63.28,47.5
3,15,370,52.41,45.2
4,20,196,49.28,42.2
5,25,32,40.64,39.1


<h1>Q8: Repeat customers</h1>

In [12]:
sql("""
WITH customer_orders AS (
    SELECT customer_id,
           COUNT(*) AS num_orders
    FROM orders
    WHERE order_status = 'Delivered'
    GROUP BY customer_id
)
SELECT COUNT(*) AS total_customers,
       SUM(CASE WHEN num_orders >= 2 THEN 1 ELSE 0 END) AS repeat_customers,
       ROUND(SUM(CASE WHEN num_orders >= 2 THEN 1 ELSE 0 END) * 100.0
             / COUNT(*), 1) AS repeat_customer_pct
FROM customer_orders
""")

,total_customers,repeat_customers,repeat_customer_pct
0,1076,669,62.2
